# CarDD Data Analysis

**Pipeline position:** 01 — Dataset inspection and exploratory analysis.

This notebook inspects the original CarDD COCO annotations. It does not convert or modify the dataset. The next notebook (02) performs the COCO → YOLO segmentation conversion.

## 1. Imports

In [ ]:
from pathlib import Path
import json
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image


## 2. Dataset Path

In [ ]:
# Update this path if the ZIP is stored elsewhere.
CARDD_ZIP = Path(r"C:\Users\Xlosn\Downloads\CarDD_release.zip")

WORKING_DIR = Path.cwd()
ZIP_PATH = CARDD_ZIP.expanduser().resolve()

print("Notebook directory:", WORKING_DIR)
print("CarDD ZIP:", ZIP_PATH)
print("Exists:", ZIP_PATH.exists())
if ZIP_PATH.exists():
    print(f"Size: {ZIP_PATH.stat().st_size / (1024**3):.2f} GB")


## 3. Inspect ZIP Structure

In [ ]:
if not ZIP_PATH.exists():
    raise FileNotFoundError("CarDD ZIP not found. Update CARDD_ZIP in the previous cell.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    names = z.namelist()

print(f"Total ZIP entries: {len(names):,}")

suffix_counts = Counter(
    Path(n).suffix.lower() for n in names if not n.endswith("/")
)
for suffix, count in suffix_counts.most_common():
    print(f"{suffix or '[none]':12s} {count:,}")


## 4. Load COCO Annotation Files

In [ ]:
# The standard CarDD release stores the three COCO split files here.
json_paths = {
    "train": "CarDD_release/CarDD_COCO/annotations/instances_train2017.json",
    "val": "CarDD_release/CarDD_COCO/annotations/instances_val2017.json",
    "test": "CarDD_release/CarDD_COCO/annotations/instances_test2017.json",
}

annotations = {}
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for split, path in json_paths.items():
        with z.open(path) as f:
            annotations[split] = json.load(f)

for split, data in annotations.items():
    print(f"{split.upper()}: images={len(data['images']):,}, annotations={len(data['annotations']):,}, categories={len(data['categories'])}")


## 5. Dataset Overview

In [ ]:
image_rows = []
annotation_rows = []
category_rows = []

for split, data in annotations.items():
    cat_map = {c["id"]: c.get("name", str(c["id"])) for c in data.get("categories", [])}

    for c in data.get("categories", []):
        category_rows.append({"split": split, "category_id": c.get("id"), "category_name": c.get("name")})

    for img in data.get("images", []):
        row = dict(img)
        row["split"] = split
        image_rows.append(row)

    for ann in data.get("annotations", []):
        row = dict(ann)
        row["split"] = split
        row["category_name"] = cat_map.get(ann.get("category_id"), str(ann.get("category_id")))
        annotation_rows.append(row)

images_df = pd.DataFrame(image_rows)
annotations_df = pd.DataFrame(annotation_rows)
categories_df = pd.DataFrame(category_rows)

print("Images:", images_df.shape)
print("Annotations:", annotations_df.shape)
print("Categories:", categories_df.shape)
display(categories_df.drop_duplicates())


## 6. Class Distribution

In [ ]:
class_counts = annotations_df["category_name"].value_counts()
display(class_counts.rename("annotation_count").to_frame())

plt.figure(figsize=(11, 6))
class_counts.sort_values().plot(kind="barh")
plt.title("Distribution of Damage Categories")
plt.xlabel("Number of annotations")
plt.ylabel("Damage category")
plt.tight_layout()
plt.show()

class_pct = (class_counts / class_counts.sum() * 100).round(2)
display(class_pct.rename("percentage").to_frame())


## 7. Unique Images per Damage Category

In [ ]:
images_per_class = (
    annotations_df.groupby("category_name")["image_id"]
    .nunique().sort_values(ascending=False)
)
display(images_per_class.rename("unique_images").to_frame())


## 8. Damage Instances per Image

In [ ]:
objects_per_image = annotations_df.groupby(["split", "image_id"]).size()
display(objects_per_image.describe().to_frame("objects_per_image"))

print("Images with 1 annotation:", int((objects_per_image == 1).sum()))
print("Images with >1 annotation:", int((objects_per_image > 1).sum()))
print("Maximum annotations in one image:", int(objects_per_image.max()))


## 9. Image Dimensions and Aspect Ratio

In [ ]:
if {"width", "height"}.issubset(images_df.columns):
    images_df["aspect_ratio"] = images_df["width"] / images_df["height"]
    images_df["image_area"] = images_df["width"] * images_df["height"]
    display(images_df[["width", "height", "aspect_ratio", "image_area"]].describe())

    plt.figure(figsize=(10, 6))
    plt.hist(images_df["aspect_ratio"].dropna(), bins=40)
    plt.title("Image Aspect Ratio Distribution")
    plt.xlabel("Width / Height")
    plt.ylabel("Number of images")
    plt.tight_layout()
    plt.show()


## 10. Bounding-Box Size Analysis

In [ ]:
bbox_records = []
for split, data in annotations.items():
    image_info = {img["id"]: {"width": img["width"], "height": img["height"]} for img in data["images"]}
    category_info = {cat["id"]: cat["name"] for cat in data["categories"]}

    for ann in data["annotations"]:
        x, y, w, h = ann["bbox"]
        W, H = image_info[ann["image_id"]]["width"], image_info[ann["image_id"]]["height"]
        bbox_records.append({
            "split": split, "image_id": ann["image_id"],
            "category": category_info[ann["category_id"]],
            "bbox_width": w, "bbox_height": h,
            "bbox_area": w * h,
            "image_width": W, "image_height": H,
            "bbox_area_ratio": (w * h) / (W * H)
        })

bbox_df = pd.DataFrame(bbox_records)
display(bbox_df[["bbox_width", "bbox_height", "bbox_area", "bbox_area_ratio"]].describe())
display(bbox_df.groupby("category")[["bbox_width", "bbox_height", "bbox_area", "bbox_area_ratio"]].median().sort_values("bbox_area_ratio"))


## 11. Handoff to Notebook 02

In [ ]:
print("01 complete: COCO dataset inspected and analyzed.")
print("Next: run 02_cardd_annotation_conversion.ipynb to create the YOLO segmentation dataset.")
